# 03 — Sealed Test Scoring + Latency/VRAM Benchmark

**Runs on:** Kaggle GPU T4. **Covers:** TASKS P7.1 and the GPU-dependent parts of P7.2.

**Run this ONCE, only after Phases 2–5 are complete** (temperatures, fusion models, scaler, conformal τ̂ all fitted and saved in the artefact repo) AND notebook 02 has produced the generalist adapter.

This is the only notebook that reads UNIFIED-TEST. The decision-layer artefacts are applied read-only here; nothing is fitted on test data. Final metric tables are computed on CPU in the evaluation step (can be here or downloaded).

In [ ]:
REPO_URL = "https://github.com/<YOUR_USER>/<YOUR_REPO>.git"  # TODO
!git clone -q {REPO_URL} slm_shield
%cd slm_shield
# Kaggle/Colab ship a pre-provisioned torch+CUDA. Do NOT `uv sync` here (it would rebuild the
# GPU stack and risk CUDA mismatch). Install the behaviour-critical libs on top of platform torch,
# pinned to the versions declared in pyproject.toml (the same ones your adapters were trained under).
!pip install -q "transformers==4.53.1" "unsloth==2025.7.2" peft trl accelerate bitsandbytes
# If Kaggle's preinstalled versions clash, restart the kernel after install and re-run from here.
from kaggle_secrets import UserSecretsClient
import os
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
ARTIFACT_REPO = "<YOUR_USER>/slm-shield-artifacts"  # TODO

In [ ]:
# --- Pull fitted decision-layer artefacts (temperatures, fusion, scaler, conformal, manifests) ---
from huggingface_hub import snapshot_download
ART = snapshot_download(ARTIFACT_REPO, repo_type="dataset", token=HF_TOKEN)
print("Artefacts at", ART)

In [ ]:
# --- Load backbone + 3 specialists + generalist ---
from src.model_loader import load_model_with_adapters
model, tokenizer = load_model_with_adapters(hf_token=HF_TOKEN, include_generalist=True)

In [ ]:
# --- P7.1: single sealed scoring pass over UNIFIED-TEST ---
from src.scoring import bulk_score_split
from src.eval.splits import load_manifests
manifests = load_manifests(f"{ART}/manifests")
test_scores = bulk_score_split(model, tokenizer, manifests, "UNIFIED-TEST",
                               out_path="/kaggle/working/scores_test.parquet", resume=True)
# Guard against the duplicated-run bug: test outputs must not be identical to any val split file.
manifests.assert_test_distinct(test_scores)

In [ ]:
# --- Latency + VRAM benchmark (P7.2 deployability metrics) ---
from src.eval.metrics import benchmark_runtime
bench = benchmark_runtime(model, tokenizer, manifests.sample("UNIFIED-TEST", n=300, seed=42),
                          modes=["batched", "sequential_no_exit", "sequential_early_exit", "legacy_generate"])
print(bench)  # ms/prompt per mode (split benign vs malicious) + peak VRAM

In [ ]:
# --- Final results matrix (CPU; applies read-only decision-layer artefacts) ---
from src.eval.run_eval import run_full_matrix
results = run_full_matrix(test_scores="/kaggle/working/scores_test.parquet",
                          artefacts_dir=ART, runtime_bench=bench,
                          out_dir="/kaggle/working/results")
print(results.summary())   # configs (a)-(f) x all metrics; also writes RESULTS_SUMMARY.md
# Persist results back to the artefact repo
from huggingface_hub import HfApi
HfApi(token=HF_TOKEN).upload_folder(folder_path="/kaggle/working/results",
                                    repo_id=ARTIFACT_REPO, repo_type="dataset",
                                    path_in_repo="results")